# CIFAR-10 Image Classification

This notebook trains a custom convolutional neural network to classify 32x32 RGB images from the CIFAR-10 benchmark dataset. The dataset contains 60,000 images across ten categories: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, and truck.

The notebook covers the full pipeline:
1. Data loading and exploration
2. Preprocessing (normalisation and one-hot encoding)
3. Model definition (three-block CNN with BatchNorm and Dropout)
4. Training with data augmentation
5. Evaluation (loss/accuracy curves, confusion matrix, classification report)
6. Inference demo on individual test images
7. Transfer learning experiment with DenseNet121

**Framework:** TensorFlow / Keras  
**Expected test accuracy:** 75-85% for the custom CNN (varies with random seed and hardware)

## 1. Install Dependencies

This cell installs required packages using subprocess so it works both in Colab and local Jupyter environments.

In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "pandas", "numpy", "matplotlib", "tensorflow", "scikit-learn"
])
print("All dependencies installed successfully.")

## 2. Imports

We import TensorFlow and Keras for model building, NumPy and Matplotlib for data handling and visualisation, and scikit-learn for evaluation metrics.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input, Dense, Conv2D, MaxPool2D, Flatten, Dropout, BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.metrics import (
    ConfusionMatrixDisplay, classification_report, confusion_matrix
)

# Suppress non-critical warnings to keep output clean
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# Set random seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")

# Detect GPU availability
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU available: {gpus[0].name}")
else:
    print("No GPU detected. Training will run on CPU.")

## 3. Load and Inspect the CIFAR-10 Dataset

CIFAR-10 is downloaded automatically by Keras on first run (approximately 170 MB). The dataset is split into 50,000 training images and 10,000 test images. Each image is 32x32 pixels with three RGB channels. Labels are integers in the range 0-9.

In [ ]:
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

print("Dataset shapes:")
print(f"  X_train: {X_train.shape}  (50,000 training images, 32x32 RGB)")
print(f"  y_train: {y_train.shape}  (50,000 integer labels)")
print(f"  X_test:  {X_test.shape}   (10,000 test images, 32x32 RGB)")
print(f"  y_test:  {y_test.shape}   (10,000 integer labels)")
print(f"\nPixel value range before normalisation: {X_train.min()} - {X_train.max()}")
print(f"Data type: {X_train.dtype}")

## 4. Exploratory Data Analysis

We visualise a random 10x10 grid of training images to get a feel for the data, then plot the class distribution to confirm the dataset is balanced.

In [ ]:
CLASS_NAMES = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]

# Display a 10x10 grid of random training images
W_grid, L_grid = 10, 10
fig, axes = plt.subplots(L_grid, W_grid, figsize=(17, 17))
axes = axes.ravel()

rng = np.random.default_rng(seed=42)
indices = rng.integers(0, len(X_train), W_grid * L_grid)

for i, index in enumerate(indices):
    axes[i].imshow(X_train[index])          # display full image (all rows intact)
    axes[i].set_title(CLASS_NAMES[int(y_train[index])], fontsize=8)
    axes[i].axis('off')

plt.suptitle('Random Sample of CIFAR-10 Training Images', fontsize=14, y=1.01)
plt.subplots_adjust(hspace=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Class distribution bar chart
classes, counts = np.unique(y_train, return_counts=True)
class_labels = [CLASS_NAMES[int(c)] for c in classes]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(class_labels, counts, color='steelblue', edgecolor='navy')
ax.set_xlabel('Number of images')
ax.set_title('Class distribution in training set')
ax.set_xlim(0, max(counts) * 1.1)

for bar, count in zip(bars, counts):
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height() / 2,
            f'{count:,}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print("The dataset is perfectly balanced: each class has exactly 5,000 training samples.")

## 5. Preprocessing

Two preprocessing steps are applied before training:

- **Normalisation**: Pixel values are scaled from the integer range [0, 255] to floating-point [0, 1]. This stabilises gradient descent.
- **One-hot encoding**: Integer class labels are converted to binary vectors of length 10, matching the 10-neuron softmax output of the model.

In [ ]:
# Normalise pixel values to [0, 1]
X_train = X_train.astype('float32') / 255.0
X_test  = X_test.astype('float32')  / 255.0

# One-hot encode labels
y_cat_train = to_categorical(y_train, num_classes=10)
y_cat_test  = to_categorical(y_test,  num_classes=10)

print(f"Pixel range after normalisation: {X_train.min():.2f} - {X_train.max():.2f}")
print(f"y_cat_train shape: {y_cat_train.shape}")
print(f"Example one-hot label (y_train[0] = {int(y_train[0])} = '{CLASS_NAMES[int(y_train[0])]}'):")
print(f"  {y_cat_train[0]}")

## 6. Model Definition

The architecture is a three-block CNN:

- **Block 1**: Conv2D(32) → BatchNorm → Conv2D(32) → BatchNorm → MaxPool → Dropout(0.25)
- **Block 2**: Conv2D(64) → BatchNorm → Conv2D(64) → BatchNorm → MaxPool → Dropout(0.25)
- **Block 3**: Conv2D(128) → BatchNorm → Conv2D(128) → BatchNorm → MaxPool → Dropout(0.25)
- **Classifier**: Flatten → Dense(128, ReLU) → Dropout(0.25) → Dense(10, Softmax)

All convolutional layers use 3x3 kernels with same-padding. Batch normalisation is applied after each Conv2D to stabilise training. Dropout regularises the network to reduce overfitting.

The model is compiled with Adam (default learning rate 0.001) and categorical cross-entropy loss. Precision and recall are tracked alongside accuracy.

In [ ]:
INPUT_SHAPE = (32, 32, 3)
KERNEL_SIZE = (3, 3)

def build_cnn():
    """Build and return the three-block CNN model."""
    model = Sequential([
        # Input layer - avoids deprecation warning from passing input_shape to Conv2D
        Input(shape=INPUT_SHAPE),

        # Block 1: 32 filters
        Conv2D(32, kernel_size=KERNEL_SIZE, activation='relu', padding='same'),
        BatchNormalization(),
        Conv2D(32, kernel_size=KERNEL_SIZE, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPool2D(pool_size=(2, 2)),
        Dropout(0.25),

        # Block 2: 64 filters
        Conv2D(64, kernel_size=KERNEL_SIZE, activation='relu', padding='same'),
        BatchNormalization(),
        Conv2D(64, kernel_size=KERNEL_SIZE, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPool2D(pool_size=(2, 2)),
        Dropout(0.25),

        # Block 3: 128 filters
        Conv2D(128, kernel_size=KERNEL_SIZE, activation='relu', padding='same'),
        BatchNormalization(),
        Conv2D(128, kernel_size=KERNEL_SIZE, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPool2D(pool_size=(2, 2)),
        Dropout(0.25),

        # Classifier head
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.25),
        Dense(10, activation='softmax'),
    ], name='cifar10_cnn')

    metrics = [
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
    ]
    model.compile(
        loss='categorical_crossentropy',
        optimizer='adam',
        metrics=metrics
    )
    return model


model = build_cnn()
model.summary()

## 7. Training with Data Augmentation

Data augmentation artificially increases the effective size of the training set by applying random transformations to each image at training time:

- **Horizontal flip**: mirrors the image left-right (valid for most CIFAR-10 classes)
- **Width/height shift**: translates the image by up to 10% in either axis

These augmentations are applied on-the-fly by `ImageDataGenerator`, meaning the original data on disk is unchanged. Augmentation is applied only to training images; test images are evaluated as-is.

Training runs for up to 50 epochs with early stopping (patience=5) to prevent overfitting. A `ReduceLROnPlateau` callback halves the learning rate if validation loss stagnates for three consecutive epochs.

In [ ]:
BATCH_SIZE = 32
EPOCHS = 50

# Data augmentation generator for training images only
data_generator = ImageDataGenerator(
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)
train_generator = data_generator.flow(X_train, y_cat_train, batch_size=BATCH_SIZE)
steps_per_epoch = X_train.shape[0] // BATCH_SIZE

# Callbacks
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

print(f"Training configuration:")
print(f"  Batch size:       {BATCH_SIZE}")
print(f"  Max epochs:       {EPOCHS}")
print(f"  Steps per epoch:  {steps_per_epoch}")
print(f"  Training samples: {X_train.shape[0]}")
print(f"  Validation samples: {X_test.shape[0]}")
print()

history = model.fit(
    train_generator,
    epochs=EPOCHS,
    steps_per_epoch=steps_per_epoch,
    validation_data=(X_test, y_cat_test),
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

## 8. Training Curves

The plots below show how loss, accuracy, precision, and recall evolved over epochs on both the training and validation sets. A gap between training and validation curves indicates some overfitting, which is expected at this model scale without techniques like label smoothing or weight decay.

In [ ]:
def plot_training_history(history):
    """Plot loss, accuracy, precision, and recall curves from training history."""
    hist = history.history
    epochs_ran = range(1, len(hist['loss']) + 1)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    metrics_to_plot = [
        ('loss',      'val_loss',      'Loss',      axes[0, 0]),
        ('accuracy',  'val_accuracy',  'Accuracy',  axes[0, 1]),
        ('precision', 'val_precision', 'Precision', axes[1, 0]),
        ('recall',    'val_recall',    'Recall',    axes[1, 1]),
    ]

    for train_key, val_key, title, ax in metrics_to_plot:
        if train_key in hist:
            ax.plot(epochs_ran, hist[train_key], label=f'Training {title}', linewidth=2)
        if val_key in hist:
            ax.plot(epochs_ran, hist[val_key], label=f'Validation {title}', linewidth=2, linestyle='--')
        ax.set_xlabel('Epoch')
        ax.set_ylabel(title)
        ax.set_title(f'{title} over epochs')
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.suptitle('Training History', fontsize=16)
    plt.tight_layout()
    plt.show()


plot_training_history(history)

## 9. Evaluation on the Test Set

We evaluate the trained model on the 10,000 held-out test images. The confusion matrix shows per-class performance: rows are true labels, columns are predicted labels. High diagonal values indicate accurate classification. The classification report provides per-class precision, recall, and F1-score.

In [ ]:
# Evaluate overall metrics on the test set
results = model.evaluate(X_test, y_cat_test, verbose=0)
metric_names = model.metrics_names

print("Test set evaluation:")
for name, value in zip(metric_names, results):
    print(f"  {name:12s}: {value:.4f}")

print()

# Generate predictions
y_pred_probs = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = y_test.flatten()

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(10, 10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(xticks_rotation='vertical', ax=ax, cmap='Blues', colorbar=True)
ax.set_title('Confusion Matrix on Test Set')
plt.tight_layout()
plt.show()

In [ ]:
# Per-class precision, recall, and F1-score
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

## 10. Inference Demo

This section demonstrates single-image inference. We display an image from the test set, run it through the model, and show the predicted class alongside the confidence score.

The grid below shows 40 test images with their predicted labels. Correct predictions are shown in blue; incorrect predictions are shown in red. A bar chart alongside each image shows the model's confidence distribution across all ten classes.

In [ ]:
# Single image inference example
index = 100
img = X_test[index]
true_class = CLASS_NAMES[int(y_test[index])]

# Run inference: model expects a batch, so reshape to (1, 32, 32, 3)
pred_probs = model.predict(img.reshape(1, 32, 32, 3), verbose=0)[0]
pred_class = CLASS_NAMES[np.argmax(pred_probs)]
confidence = np.max(pred_probs) * 100

fig, axes = plt.subplots(1, 2, figsize=(8, 3))

axes[0].imshow(img)
axes[0].axis('off')
color = 'blue' if pred_class == true_class else 'red'
axes[0].set_title(
    f"Predicted: {pred_class} ({confidence:.1f}%)\nTrue: {true_class}",
    color=color, fontsize=11
)

bars = axes[1].bar(range(10), pred_probs, color='#777777')
bars[np.argmax(pred_probs)].set_color('red')
bars[int(y_test[index])].set_color('blue')
axes[1].set_xticks(range(10))
axes[1].set_xticklabels(CLASS_NAMES, rotation=45, ha='right', fontsize=8)
axes[1].set_ylabel('Confidence')
axes[1].set_ylim([0, 1])
axes[1].set_title('Prediction confidence\n(red=predicted, blue=true)')
axes[1].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
def plot_image(i, predictions_array, true_labels, images):
    """Display a single image with its predicted and true label."""
    pred_array = predictions_array
    true_label = int(true_labels[i])
    img = images[i]

    plt.grid(False)
    plt.xticks([])
    plt.yticks([])
    plt.imshow(img)

    predicted_label = np.argmax(pred_array)
    color = 'blue' if predicted_label == true_label else 'red'
    plt.xlabel(
        f"{CLASS_NAMES[predicted_label]} {100 * np.max(pred_array):.0f}%"
        f" ({CLASS_NAMES[true_label]})",
        color=color
    )


def plot_value_array(i, predictions_array, true_labels):
    """Display a bar chart of prediction confidence for a single image."""
    true_label = int(true_labels[i])
    plt.grid(False)
    plt.xticks(range(10))
    plt.yticks([])
    bars = plt.bar(range(10), predictions_array, color='#777777')
    plt.ylim([0, 1])
    predicted_label = np.argmax(predictions_array)
    bars[predicted_label].set_color('red')
    bars[true_label].set_color('blue')


# Plot a grid of 40 test images with predictions
num_rows = 8
num_cols = 5
num_images = num_rows * num_cols

plt.figure(figsize=(2 * 2 * num_cols, 2 * num_rows))
for i in range(num_images):
    plt.subplot(num_rows, 2 * num_cols, 2 * i + 1)
    plot_image(i, y_pred_probs[i], y_test, X_test)
    plt.subplot(num_rows, 2 * num_cols, 2 * i + 2)
    plot_value_array(i, y_pred_probs[i], y_test)

plt.suptitle(
    'First 40 test images with predictions (blue=correct, red=incorrect)',
    fontsize=13
)
plt.tight_layout()
plt.show()

### Random Test Image Predictions

The following grid displays 25 randomly selected test images with the model's predicted labels. This gives a qualitative sense of where the model succeeds and where it makes errors (common confusions include cat/dog and automobile/truck).

In [ ]:
W_grid, L_grid = 5, 5
fig, axes = plt.subplots(L_grid, W_grid, figsize=(13, 13))
axes = axes.ravel()

rng2 = np.random.default_rng(seed=99)
test_indices = rng2.integers(0, len(X_test), W_grid * L_grid)

for i, index in enumerate(test_indices):
    axes[i].imshow(X_test[index])
    pred_label = CLASS_NAMES[int(y_pred[index])]
    true_label = CLASS_NAMES[int(y_test[index])]
    correct = y_pred[index] == int(y_test[index])
    title_color = 'green' if correct else 'red'
    axes[i].set_title(f"Pred: {pred_label}\nTrue: {true_label}",
                      fontsize=8, color=title_color)
    axes[i].axis('off')

plt.suptitle('Random test image predictions (green=correct, red=incorrect)',
             fontsize=12)
plt.subplots_adjust(hspace=0.5)
plt.tight_layout()
plt.show()

## 11. Transfer Learning with DenseNet121

As a secondary experiment we fine-tune DenseNet121 pre-trained on ImageNet. The base model's convolutional layers are frozen (weights come from ImageNet pre-training), and only the final Dense(10) classification head is trained.

DenseNet121 expects images of at least 32x32, so CIFAR-10 images fit without upscaling, though the architecture was designed for much larger inputs. The `pooling='avg'` argument replaces the original classification head with global average pooling before our dense layer.

**Note**: This experiment uses the same augmented training generator from the previous section. DenseNet121 typically converges slower on CIFAR-10 at this resolution compared to a custom CNN, but can achieve higher accuracy given enough epochs.

In [ ]:
from tensorflow.keras.applications import DenseNet121

def build_densenet_model():
    """Build a DenseNet121-based model with a custom classification head."""
    base_model = DenseNet121(
        input_shape=(32, 32, 3),
        include_top=False,
        weights='imagenet',
        pooling='avg'
    )
    # Freeze base model weights for feature extraction
    base_model.trainable = False

    transfer_model = Sequential([
        base_model,
        Dense(10, activation='softmax')
    ], name='densenet121_cifar10')

    transfer_model.compile(
        loss='categorical_crossentropy',
        optimizer='adam',
        metrics=['accuracy']
    )
    return transfer_model


transfer_model = build_densenet_model()
transfer_model.summary()

In [ ]:
# Recreate the generator to ensure it has not been exhausted
transfer_generator = ImageDataGenerator(
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
).flow(X_train, y_cat_train, batch_size=BATCH_SIZE)

transfer_early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

print("Training DenseNet121 transfer learning model...")
print("Note: Only the Dense(10) head is being trained; base weights are frozen.")
print()

transfer_history = transfer_model.fit(
    transfer_generator,
    epochs=20,
    steps_per_epoch=steps_per_epoch,
    validation_data=(X_test, y_cat_test),
    callbacks=[transfer_early_stop],
    verbose=1
)

In [ ]:
# Evaluate transfer learning model
transfer_results = transfer_model.evaluate(X_test, y_cat_test, verbose=0)
print(f"DenseNet121 transfer model - Test accuracy: {transfer_results[1] * 100:.2f}%")

# Compare with custom CNN
cnn_acc = results[1]
print(f"Custom CNN             - Test accuracy: {cnn_acc * 100:.2f}%")

# Plot transfer learning training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(transfer_history.history['loss'], label='Training loss')
axes[0].plot(transfer_history.history['val_loss'], label='Validation loss', linestyle='--')
axes[0].set_title('DenseNet121 Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(transfer_history.history['accuracy'], label='Training accuracy')
axes[1].plot(transfer_history.history['val_accuracy'], label='Validation accuracy', linestyle='--')
axes[1].set_title('DenseNet121 Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('DenseNet121 Transfer Learning Training Curves')
plt.tight_layout()
plt.show()

## 12. Summary

This notebook demonstrated end-to-end image classification on CIFAR-10 using TensorFlow/Keras.

**Key results:**

| Model | Test Accuracy |
|-------|---------------|
| Custom 3-block CNN | 75-85% |
| DenseNet121 (head only) | 60-75% |

**Notes on results:**
- The custom CNN achieves competitive accuracy for its parameter count (~550K trainable parameters).
- DenseNet121 in head-only mode performs below the custom CNN at this resolution because the ImageNet pre-trained features are not well-matched to 32x32 images. Fine-tuning more layers or upscaling images would likely improve transfer learning performance.
- Common misclassifications occur between visually similar classes: cat/dog, automobile/truck, and bird/airplane.

**Further improvements that could raise accuracy:**
- Stronger augmentation (random crops, colour jitter, cutout)
- Weight decay (L2 regularisation) on convolutional layers
- Residual connections (as in ResNet)
- Label smoothing
- Cosine annealing learning rate schedule